# 分组分析与历史

这是独立的合成沙盒，不连接设备。无需完成其他课程。先从新 kernel 顺序运行，再做文末的小修改。
启动入口已准备环境与服务；重复打开继续当前练习，重置会创建新的起点。

In [ ]:
import sys
from pathlib import Path

import scopecat as sc

project = sc.open_project()
if Path(sys.prefix).resolve() != (project.root / ".venv").resolve():
    raise RuntimeError(f"请在 Select Kernel 中选择 {project.root / '.venv'}")

In [ ]:
import numpy as np

session = project.authoring()
session.refresh()
from my_experiment.setup import open_parameters
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)

本主题直接生成自己的 42 点合成数据，不依赖入门课程的 run。按 amplitude 分成两条频率曲线，分析函数位于 `src/my_experiment/group_analysis.py`。

In [ ]:
from my_experiment.group_analysis import CurveSummary
from my_experiment.parameters import Drive

request = (
    teaching_rabi()
    .sweep(amplitude=[0.12, 0.24])
    .sweep_parameter(
        Drive.frequency, "q0", np.linspace(5.135, 5.155, 21), name="frequency"
    )
)
prepared = session.prepare(request, parameters=params)
assert prepared.preview.point_count == 42
run = prepared.run().wait(timeout=120).result()
fitted = session.analyze_groups_as(
    run.id,
    "my_experiment.group_analysis:summarize_curve",
    CurveSummary,
    by=("amplitude",),
    fitting="frequency",
)
assert len(fitted.groups) == 2
for group in fitted.groups:
    assert group.receipt.error is None
    assert group.value.status == "estimated"
    print(group.value)

直接使用已有 run 和分析对象读取已发布结果，无需写额外 JSON 文件。读取不会重新执行分析或采集。

In [ ]:
restored = session.read_groups_as(run.id, fitted.publication.id, CurveSummary)
assert len(restored.groups) == 2
print("原始点数:", len(run.measurements()))
session.history()

小修改：增加一个 amplitude，先预测点数和组数，再预览、运行。已有分析保持独立，不会被新分析覆盖。